In [10]:
import sys
sys.path.append("..")

import pandas as pd

short_ic_ranking = pd.read_excel("../exports/RANKING_short_iron_condor.xlsx")

# Filter out gap-risk-flagged AND volatility-persistence-flagged stocks
clean_candidates = short_ic_ranking[
    (short_ic_ranking["gap_risk_flag"].isna() | (short_ic_ranking["gap_risk_flag"] == "")) &
    (short_ic_ranking["volatility_persistence_flag"].isna() | (short_ic_ranking["volatility_persistence_flag"] == ""))
]
top_25 = clean_candidates.head(25)

must_include = short_ic_ranking[short_ic_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[
    (must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")) &
    (must_include["volatility_persistence_flag"].isna() | (must_include["volatility_persistence_flag"] == ""))
]

final_candidates = pd.concat([top_25, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates

,symbol,short_ic_score,gap_risk_flag,Total flips (closed),Median monthly range_pct,Largest monthly range_pct,Median max_adverse_pct,Worst max_adverse_pct (4yr),Median duration (days),Worst single gap (4yr),last_month_range_pct,volatility_persistence_flag
0,NIFTY,97.6,NaN,76,5.36,12.39,1.06,5.95,15.2,2.32,3.87,NaN
1,SENSEX,97.1,NaN,76,5.51,12.32,1.08,5.55,15.6,2.31,4.17,NaN
2,FINNIFTY,96.8,NaN,80,6.20,15.57,1.28,7.04,14.2,2.35,5.48,NaN
3,MIDCPNIFTY,95.7,NaN,80,7.02,16.90,1.87,7.40,14.0,2.84,4.45,NaN
4,NIFTYNXT50,92.0,NaN,82,7.57,18.74,1.23,9.24,15.1,2.80,4.47,NaN
5,BANKNIFTY,88.2,NaN,69,6.20,17.01,1.52,7.15,19.0,3.73,4.47,NaN
6,GRASIM,84.7,NaN,65,9.02,21.67,2.42,6.95,18.2,3.44,5.97,NaN
7,SUNPHARMA,83.8,NaN,53,8.48,16.84,2.57,7.31,20.0,3.53,9.94,NaN
8,DABUR,83.3,NaN,65,8.59,21.12,2.41,7.34,19.2,2.66,9.56,NaN
9,ULTRACEMCO,83.1,NaN,66,8.52,22.72,2.39,7.21,19.0,3.61,8.55,NaN


In [11]:
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.fetch_data import fetch_historical_data, save_raw_data
from src.supertrend import apply_supertrend
from src.flip_analysis import detect_flips
from src.monthly_range import compute_monthly_range
from src.summary_stats import compute_summary_stats
from src.export_report import export_full_report
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        df = fetch_historical_data(kite, token, from_date=datetime(2022, 1, 1), to_date=datetime.now(), interval="4hour")
        save_raw_data(df, symbol)
        df = df.set_index("date")

        df_st = apply_supertrend(df)
        trades = detect_flips(df_st)
        monthly = compute_monthly_range(df)
        summary = compute_summary_stats(trades, monthly)
        export_full_report(trades, monthly, summary, symbol)

        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-03...
Saved 2259 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NIFTY_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/NIFTY_full_report.xlsx
[1/25] NIFTY — refreshed
[2/25] SENSEX — no token found, skipping
[3/25] FINNIFTY — no token found, skipping
[4/25] MIDCPNIFTY — no token found, skipping
[5/25] NIFTYNXT50 — no token found, skipping


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-03...
Saved 2259 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/BANKNIFTY_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/BANKNIFTY_full_report.xlsx
[6/25] BANKNIFTY — refreshed
Fetching 2022-01-01 to 2023-01-31...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-03...
Saved 2259 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/GRASIM_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/GRASIM_full_report.xlsx
[7/25] GRASIM — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...


...

/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-03...
Saved 2259 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/OFSS_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/OFSS_full_report.xlsx
[24/25] OFSS — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-03...
Saved 2259 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ABB_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/ABB_full_report.xlsx
[25/25] ABB — refreshed


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


In [12]:
from src.monthly_range_screener import screen_monthly_range

live_check = screen_monthly_range(symbols_to_refresh)
live_check.sort_values("current_month_range_pct", ascending=False)

,symbol,days_into_month,current_month_range_pct,median_monthly_range,75th_pct_monthly_range,90th_pct_monthly_range,already_past_median,already_past_75th,already_past_90th
15,GAIL,3,5.62,12.66,16.70,20.12,False,False,False
20,ABB,3,5.12,13.28,18.02,24.21,False,False,False
12,SHREECEM,3,3.10,9.86,13.73,17.21,False,False,False
9,EICHERMOT,3,2.58,11.73,14.70,18.05,False,False,False
3,SUNPHARMA,3,2.57,8.48,11.18,13.42,False,False,False
7,COLPAL,3,2.22,9.00,11.94,18.98,False,False,False
16,VOLTAS,3,2.15,12.26,16.89,23.97,False,False,False
19,OFSS,3,2.13,12.27,19.13,23.88,False,False,False
11,JSWSTEEL,3,2.04,10.28,12.98,15.26,False,False,False
2,GRASIM,3,1.88,9.02,11.53,14.90,False,False,False
